# E-commerce Medallion Architecture Assignment

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType, IntegerType, DateType

## Part A - Environment & Bronze Layer

### Q1

In [0]:
# current Spark version, catalog, and schema
print(f"Spark version: {spark.version}")
print(f"Current catalog: {spark.catalog.currentCatalog()}")
print(f"Current schema: {spark.catalog.currentDatabase()}")

Spark version: 4.1.0
Current catalog: workspace
Current schema: default


### Q2

ecom_bronze, ecom_silver, and ecom_gold all created under the workspace catalog

ecom_bronze is where the original data sources are first loaded into before performing so quick statistics of the data.
ecom_silver is where the bronze data (which might have some transformed columns from the source) will be evaluated, cleaned, and saved to.
ecom_gold is where tables related to analytical insights will live using the cleaned and validated data from ecom_silver.

### Q3

ecommerce_training_files was created under the default schema in the workspace catalog.

In [0]:
# show volume path
VOLUME_PATH = "/Volumes/workspace/default/ecommorce_training_files/"
print("Volume path:", VOLUME_PATH)

Volume path: /Volumes/workspace/default/ecommorce_training_files/


### Q4

Verifying the existence of all seven csv files in the volume can be done within the Databricks Catalog menu. However, we can also display all of the files in the volume programmatically.

In [0]:
raw_files = [f.name for f in dbutils.fs.ls(VOLUME_PATH)]
print(f"Files in volume ({len(raw_files)}):")
for file in raw_files:
    df = pd.read_csv(f"{VOLUME_PATH}/{file}")
    print(f"{file}: {len(df)} rows")

Files in volume (7):
customers.csv: 6000 rows
order_items.csv: 62053 rows
orders.csv: 25000 rows
payments.csv: 25000 rows
products.csv: 1200 rows
returns.csv: 3000 rows
shipments.csv: 20000 rows


### Q5

printSchema shows all columns are read as a string.

In [0]:
orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", "false")
    .csv(f"{VOLUME_PATH}/orders.csv")
)

orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- subtotal: string (nullable = true)
 |-- discount_amount: string (nullable = true)
 |-- tax_amount: string (nullable = true)
 |-- shipping_cost: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- promo_code: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- shipping_state: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- last_updated: string (nullable = true)



### Q6

In [0]:
# reusable ingestion function
def ingest_bronze(file_name: str):
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(file_name)
    )

    columns = df.columns

    df = df.select(
        *columns,
        F.lit(file_name.split("/")[-1]).alias("_source_file"),
        F.current_timestamp().alias("_ingested_at"),
        F.sha2(F.concat_ws("||", *[F.col(c) for c in columns]), 256).alias("_raw_row_hash")
    )

    return df

### Q7

In [0]:
# use ingestion function, write delta table for each raw file
FILE_TO_ENTITY = {
    "customers.csv": "customers_raw",
    "products.csv": "products_raw",
    "orders.csv": "orders_raw",
    "order_items.csv": "order_items_raw",
    "payments.csv": "payments_raw",
    "shipments.csv": "shipments_raw",
    "returns.csv": "returns_raw",
}

for file_name, table_name in FILE_TO_ENTITY.items():
    df = ingest_bronze(f"{VOLUME_PATH}/{file_name}")
    df.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(f"workspace.ecom_bronze.{table_name}")

### Q8

In [0]:
rows = []
for table_name in FILE_TO_ENTITY.values():
    full_name = f"workspace.ecom_bronze.{table_name}"
    df = spark.read.table(full_name)
    rows.append((table_name, df.count(), len(df.columns)))

profile_df = spark.createDataFrame(
    rows,
    ["table_name", "rows", "columns"]
)
profile_df.show(truncate = False)

+---------------+-----+-------+
|table_name     |rows |columns|
+---------------+-----+-------+
|customers_raw  |6000 |18     |
|products_raw   |1200 |15     |
|orders_raw     |25000|19     |
|order_items_raw|62053|10     |
|payments_raw   |25000|10     |
|shipments_raw  |20000|11     |
|returns_raw    |3000 |10     |
+---------------+-----+-------+



### Q9

In [0]:
for table_name in FILE_TO_ENTITY.values():
    df = spark.read.table(f"workspace.ecom_bronze.{table_name}")
    dupes = (
        df.groupBy("_raw_row_hash")
        .count()
        .filter("count > 1")
    )
    print(table_name, "duplicate raw rows:", dupes.count())

customers_raw duplicate raw rows: 0
products_raw duplicate raw rows: 0
orders_raw duplicate raw rows: 0
order_items_raw duplicate raw rows: 100
payments_raw duplicate raw rows: 200
shipments_raw duplicate raw rows: 0
returns_raw duplicate raw rows: 0


### Q10

In [0]:
# customers
customers = spark.read.table("workspace.ecom_bronze.customers_raw")

customer_nulls = customers.select(
    (F.count(F.when(F.col("customer_id").isNull() | (F.col("customer_id") == ""), 1))).alias("customer_id_nulls"),
    (F.count(F.when(F.col("first_name").isNull() | (F.col("first_name") == ""), 1))).alias("first_name_nulls"),
    (F.count(F.when(F.col("email").isNull() | (F.col("email") == ""), 1))).alias("email_nulls"),
    (F.count(F.when(F.col("date_of_birth").isNull() | (F.col("date_of_birth") == ""), 1))).alias("date_of_birth_nulls")
)

customer_nulls.show(truncate = False)

+-----------------+----------------+-----------+-------------------+
|customer_id_nulls|first_name_nulls|email_nulls|date_of_birth_nulls|
+-----------------+----------------+-----------+-------------------+
|0                |52              |0          |0                  |
+-----------------+----------------+-----------+-------------------+



In [0]:
# order_items
order_items = spark.read.table("workspace.ecom_bronze.order_items_raw")

order_items_nulls = order_items.select(
    (F.count(F.when(F.col("order_item_id").isNull() | (F.col("order_item_id") == ""), 1))).alias("order_item_id_nulls"),
    (F.count(F.when(F.col("order_id").isNull() | (F.col("order_id") == ""), 1))).alias("order_id_nulls"),
    (F.count(F.when(F.col("product_id").isNull() | (F.col("product_id") == ""), 1))).alias("product_id_nulls"),
    (F.count(F.when(F.col("unit_price").isNull() | (F.col("unit_price") == ""), 1))).alias("unit_price_nulls")
)

order_items_nulls.show(truncate = False)

+-------------------+--------------+----------------+----------------+
|order_item_id_nulls|order_id_nulls|product_id_nulls|unit_price_nulls|
+-------------------+--------------+----------------+----------------+
|0                  |0             |0               |0               |
+-------------------+--------------+----------------+----------------+



In [0]:
# orders
orders = spark.read.table("workspace.ecom_bronze.orders_raw")

order_nulls = orders.select(
    (F.count(F.when(F.col("order_id").isNull() | (F.col("order_id") == ""), 1))).alias("order_id_nulls"),
    (F.count(F.when(F.col("customer_id").isNull() | (F.col("customer_id") == ""), 1))).alias("customer_id_nulls"),
    (F.count(F.when(F.col("order_date").isNull() | (F.col("order_date") == ""), 1))).alias("order_date_nulls"),
    (F.count(F.when(F.col("order_status").isNull() | (F.col("order_status") == ""), 1))).alias("order_status_nulls")
)

order_nulls.show(truncate = False)

+--------------+-----------------+----------------+------------------+
|order_id_nulls|customer_id_nulls|order_date_nulls|order_status_nulls|
+--------------+-----------------+----------------+------------------+
|0             |103              |0               |0                 |
+--------------+-----------------+----------------+------------------+



In [0]:
# payments
payments = spark.read.table("workspace.ecom_bronze.payments_raw")

payment_nulls = payments.select(
    (F.count(F.when(F.col("payment_id").isNull() | (F.col("payment_id") == ""), 1))).alias("payment_id_nulls"),
    (F.count(F.when(F.col("order_id").isNull() | (F.col("order_id") == ""), 1))).alias("order_id_nulls"),
    (F.count(F.when(F.col("payment_date").isNull() | (F.col("payment_date") == ""), 1))).alias("payment_date_nulls"),
    (F.count(F.when(F.col("transaction_id").isNull() | (F.col("transaction_id") == ""), 1))).alias("transaction_id_nulls")
)

payment_nulls.show(truncate = False)

+----------------+--------------+------------------+--------------------+
|payment_id_nulls|order_id_nulls|payment_date_nulls|transaction_id_nulls|
+----------------+--------------+------------------+--------------------+
|0               |0             |0                 |0                   |
+----------------+--------------+------------------+--------------------+



In [0]:
# products 
products = spark.read.table("workspace.ecom_bronze.products_raw")

product_nulls = products.select(
    (F.count(F.when(F.col("product_id").isNull() | (F.col("product_id") == ""), 1))).alias("product_id_nulls"),
    (F.count(F.when(F.col("category").isNull() | (F.col("category") == ""), 1))).alias("category_nulls"),
    (F.count(F.when(F.col("unit_price").isNull() | (F.col("unit_price") == ""), 1))).alias("unit_price_nulls"),
    (F.count(F.when(F.col("supplier_id").isNull() | (F.col("supplier_id") == ""), 1))).alias("supplier_id_nulls")
)

product_nulls.show(truncate = False)

+----------------+--------------+----------------+-----------------+
|product_id_nulls|category_nulls|unit_price_nulls|supplier_id_nulls|
+----------------+--------------+----------------+-----------------+
|0               |16            |0               |0                |
+----------------+--------------+----------------+-----------------+



In [0]:
# returns
returns = spark.read.table("workspace.ecom_bronze.returns_raw")

return_nulls = returns.select(
    (F.count(F.when(F.col("return_id").isNull() | (F.col("return_id") == ""), 1))).alias("return_id_nulls"),
    (F.count(F.when(F.col("order_id").isNull() | (F.col("order_id") == ""), 1))).alias("order_id_nulls"),
    (F.count(F.when(F.col("return_date").isNull() | (F.col("return_date") == ""), 1))).alias("return_date_nulls"),
    (F.count(F.when(F.col("return_status").isNull() | (F.col("return_status") == ""), 1))).alias("return_status_nulls")
)

return_nulls.show(truncate = False)

+---------------+--------------+-----------------+-------------------+
|return_id_nulls|order_id_nulls|return_date_nulls|return_status_nulls|
+---------------+--------------+-----------------+-------------------+
|0              |0             |0                |0                  |
+---------------+--------------+-----------------+-------------------+



In [0]:
# shipments 
shipments = spark.read.table("workspace.ecom_bronze.shipments_raw")

shipment_nulls = shipments.select(
    (F.count(F.when(F.col("shipment_id").isNull() | (F.col("shipment_id") == ""), 1))).alias("shipment_id_nulls"),
    (F.count(F.when(F.col("shipment_date").isNull() | (F.col("shipment_date") == ""), 1))).alias("shipment_date_nulls"),
    (F.count(F.when(F.col("tracking_number").isNull() | (F.col("tracking_number") == ""), 1))).alias("tracking_number_nulls"),
    (F.count(F.when(F.col("shipment_status").isNull() | (F.col("shipment_status") == ""), 1))).alias("shipment_status_nulls")
)

shipment_nulls.show(truncate = False)

+-----------------+-------------------+---------------------+---------------------+
|shipment_id_nulls|shipment_date_nulls|tracking_number_nulls|shipment_status_nulls|
+-----------------+-------------------+---------------------+---------------------+
|0                |0                  |111                  |0                    |
+-----------------+-------------------+---------------------+---------------------+



## Part B - Silver Data Quality and Transformations

### Q11

In [0]:
customers = (
    spark.table("workspace.ecom_bronze.customers_raw")
    .withColumn("first_name", F.initcap(F.trim(F.col("first_name"))))
    .withColumn("last_name",  F.initcap(F.trim(F.col("last_name"))))
    .withColumn("email",      F.lower(F.trim(F.col("email"))))
    .withColumn("city",       F.initcap(F.trim(F.col("city"))))
    .withColumn("state",      F.initcap(F.trim(F.col("state"))))
    .withColumn("country",    F.initcap(F.trim(F.col("country"))))
)

### Q12

In [0]:
CUSTOMER_ID_RE = r"^C[0-9]{6}$"
EMAIL_RE = r"^[^@\s]+@[^@\s]+\.[A-Za-z]{2,}$"

customers = (
    customers
    .withColumn("is_valid_id", F.col("customer_id").rlike(CUSTOMER_ID_RE))
    .withColumn("is_valid_email", F.col("email").rlike(EMAIL_RE))
)

In [0]:
customers = (
    customers
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(~F.col("is_valid_id"), F.lit("invalid_id")),
            F.when(~F.col("is_valid_email"), F.lit("invalid_email"))
        )
    )
)

### Q13

In [0]:
customers = (
    customers
    .withColumn("date_of_birth_clean", F.try_to_date(F.col("date_of_birth"), "yyyy-MM-dd"))
    .withColumn("signup_date_clean", F.try_to_date(F.col("signup_date"), "yyyy-MM-dd"))
)

In [0]:
customers = (
    customers
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("dq_reason") == "", F.lit(None)).otherwise(F.col("dq_reason")),
            F.when(F.col("date_of_birth_clean").isNull(), F.lit("invalid_date_of_birth")),
            F.when(F.col("signup_date_clean").isNull(), F.lit("invalid_signup_date"))
        )
    )
)

### Q14

In [0]:
customers = (
    customers
    .withColumn("_is_active_norm", F.lower(F.trim(F.col("is_active"))))
    .withColumn(
        "is_active",
        F.when(F.col("_is_active_norm").isin("y", "yes", "1", "true"), F.lit(True))
         .when(F.col("_is_active_norm").isin("n", "no", "0", "false"), F.lit(False))
         .otherwise(F.lit(None))
    )
    .drop("_is_active_norm")
)

### Q15

In [0]:
customers = (
    customers
    .withColumn("loyalty_tier_norm", F.trim(F.lower(F.col("loyalty_tier"))))
    .withColumn(
        "gender",
        F.when(F.trim(F.lower(F.col("gender"))).isin(["m", "male"]), F.lit("male"))
        .when(F.trim(F.lower(F.col("gender"))).isin(["f", "female"]), F.lit("female"))
        .when(F.trim(F.lower(F.col("gender"))).isin(["o", "other"]), F.lit("other"))
        .otherwise(F.lit(None))
    )
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("dq_reason") == "", F.lit(None)).otherwise(F.col("dq_reason")),
            F.when(~F.col("loyalty_tier_norm").isin(["bronze", "silver", "gold", "platinum"]), F.lit("invalid_loyalty_tier")))
        )
    .withColumn("loyalty_tier", F.col("loyalty_tier_norm"))
    .drop("loyalty_tier_norm")
)

### Q16

In [0]:
w = Window.partitionBy("customer_id").orderBy(F.desc_nulls_last("signup_date_clean"))

deduped_customers = (
    customers
    .withColumn("row_num", F.row_number().over(w))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

### Q17

In [0]:
# create tables 
customers = (
    deduped_customers
    .filter((F.col("dq_reason").isNull()) | (F.length(F.col("dq_reason")) == 0))
)

customers_quarantine = (
    deduped_customers
    .filter((F.col("dq_reason").isNotNull()) & (F.length(F.col("dq_reason")) > 0))
)

# save corresponding delta tables
customers.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.customers")
customers_quarantine.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.customers_quarantine")

# show five most common quarantine reasons
display(customers_quarantine.groupBy("dq_reason").count().orderBy(F.desc("count")).limit(5))

dq_reason,count
invalid_email,124
invalid_signup_date,72
invalid_loyalty_tier,37
invalid_id,34
"invalid_email, invalid_signup_date",1


### Q18

In [0]:
# cast right types
products = (
    products
    .withColumn("unit_price", F.col("unit_price").try_cast(DecimalType(18, 2)))
    .withColumn("cost_price", F.col("cost_price").try_cast(DecimalType(18, 2)))
    .withColumn("stock_quantity", F.col("stock_quantity").try_cast(IntegerType()))
    .withColumn("product_rating", F.col("product_rating").try_cast(DecimalType(4, 1)))
    .withColumn("created_date", F.col("created_date").try_cast(DateType()))
)

### Q19

In [0]:
ALLOWED_CATEGORIES = ["Grocery", "Sports", "Electronics", "Books", "Beauty", "Fashion", "Home"]

products = (
    products
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when((F.col("unit_price").isNull()) | (F.col("unit_price") <= 0), F.lit("invalid_unit_price")),
            F.when((F.col("cost_price").isNull()) | (F.col("cost_price") <= 0), F.lit("invalid_cost_price")),
            F.when((F.col("stock_quantity").isNull()) | (F.col("stock_quantity") < 0), F.lit("invalid_stock_quantity")),
            F.when((F.col("product_rating").isNull())| (~F.col("product_rating").between(0, 5)), F.lit("invalid_product_rating")),
            F.when(
                F.col("category").isNull() | (F.trim(F.col("category")) == "") | (~F.col("category").isin(ALLOWED_CATEGORIES)),
                F.lit("invalid_category")
            ),
            F.when(F.col("cost_price") > F.col("unit_price"), F.lit("cost_larger_than_price"))
        )
    )
)

### Q20

In [0]:
w = Window.partitionBy("product_id").orderBy(F.desc_nulls_last("created_date"))

deduped_products = (
    products
    .withColumn("row_num", F.row_number().over(w))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [0]:
# create tables 
products = (
    deduped_products
    .filter((F.col("dq_reason").isNull()) | (F.length(F.col("dq_reason")) == 0))
)

products_quarantine = (
    deduped_products
    .filter((F.col("dq_reason").isNotNull()) & (F.length(F.col("dq_reason")) > 0))
)

# save corresponding delta tables
products.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.products")
products_quarantine.write.format("delta").mode("overwrite").saveAsTable("workspace.ecom_silver.products_quarantine")

### Q21

In [0]:
# multiple timestamp formats
orders = (
    orders
    .withColumn(
        "order_date_clean",
        F.coalesce(
            F.expr("try_to_timestamp(order_date, 'yyyy-MM-dd HH:mm:ss')"),
            F.expr("try_to_timestamp(order_date, 'MM/dd/yyyy HH:mm')"),
            F.expr("try_to_timestamp(order_date, 'yyyy/MM/dd HH:mm:ss')"),
            F.expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
        )
    )
    .withColumn(
        "last_updated_clean",
        F.coalesce(
            F.expr("try_to_timestamp(last_updated, 'yyyy-MM-dd HH:mm:ss')"),
            F.expr("try_to_timestamp(last_updated, 'MM/dd/yyyy HH:mm')"),
            F.expr("try_to_timestamp(last_updated, 'yyyy/MM/dd HH:mm:ss')"),
            F.expr("try_to_timestamp(last_updated, 'yyyy-MM-dd')"),
        )
    )
    .drop("order_date", "last_updated")
    .withColumnRenamed("order_date_clean", "order_date")
    .withColumnRenamed("last_updated_clean", "last_updated")
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("order_date").isNull(), F.lit("invalid_order_date")),
            F.when(F.col("last_updated").isNull(), F.lit("invalid_last_updated")),
        )
    )
)

### Q22

In [0]:
status = F.lower(F.trim(F.col("order_status")))

orders = (
    orders
    .withColumn(
        "order_status",
        F.when(status == "complete", F.lit("delivered"))
        .when(status == "in-transit", F.lit("shipped"))
        .when(status.isin(
            "placed", "pending", "processing", "shipped", "delivered", "cancelled", "returned"
        ), status)
    )
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("dq_reason") == "", F.lit(None)).otherwise(F.col("dq_reason")),
            F.when(F.col("order_status").isNull(), F.lit("invalid_order_status")),
        )
    )
)

### Q23

In [0]:
orders = (
    orders
    .withColumn("channel", F.lower(F.trim(F.col("channel"))))
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("dq_reason") == "", F.lit(None)).otherwise(F.col("dq_reason")),
            F.when(F.col("channel").isNull(), F.lit("invalid_channel")),
            F.when(F.col("currency").isNull(), F.lit("invalid_currency")),
        )
    )
)

### Q24

In [0]:
orders = (
    orders
    .withColumn("subtotal", F.col("subtotal").try_cast(DecimalType(18, 2)))
    .withColumn("discount_amount", F.col("discount_amount").try_cast(DecimalType(18, 2)))
    .withColumn("tax_amount", F.col("tax_amount").try_cast(DecimalType(18, 2)))
    .withColumn("shipping_cost", F.col("shipping_cost").try_cast(DecimalType(18, 2)))
    .withColumn("total_amount", F.col("total_amount").try_cast(DecimalType(18, 2)))
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("dq_reason") == "", F.lit(None)).otherwise(F.col("dq_reason")),
            F.when(F.col("subtotal").isNull(), F.lit("invalid_subtotal")),
            F.when(F.col("discount_amount").isNull(), F.lit("invalid_discount_amount")),
            F.when(F.col("tax_amount").isNull(), F.lit("invalid_tax_amount")),
            F.when(F.col("shipping_cost").isNull(), F.lit("invalid_shipping_cost")),
            F.when(F.col("total_amount").isNull(), F.lit("invalid_total_amount"))
        )
    )
)

### Q25

In [0]:
# validate financial columns
expected = (F.col("subtotal") - F.col("discount_amount") + F.col("tax_amount") + F.col("shipping_cost"))
financials_ok = F.abs(F.col("total_amount") - expected) <= F.lit(1.00)

orders = (
    orders
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("dq_reason") == "", F.lit(None)).otherwise(F.col("dq_reason")),
            F.when(~financials_ok, F.lit("invalid_financials"))
        )
    )
)

### Q26

In [0]:
customer_keys = (
    spark.table("workspace.ecom_silver.customers")
    .select("customer_id")
    .distinct()
)

orders = (
    orders.join(
        F.broadcast(customer_keys.withColumn("_cust_ok", F.lit(True))),
        on = "customer_id",
        how = "left",
    )
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("dq_reason") == "", F.lit(None)).otherwise(F.col("dq_reason")),
            F.when(F.col("_cust_ok").isNull(), F.lit("missing_customer")),
        )
    )
    .drop("_cust_ok")
)

### Q27

In [0]:
w = Window.partitionBy("order_id").orderBy(F.desc_nulls_last("last_updated"))

deduped_orders = (
    orders
    .withColumn("row_num", F.row_number().over(w))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [0]:
# save clean and quarantine tables
(
    deduped_orders
    .filter(F.col("dq_reason").isNull())
    .write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.ecom_silver.orders")
)

(
    deduped_orders
    .filter(F.col("dq_reason").isNotNull())
    .write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.ecom_silver.orders_quarantine")
)

### Q28

In [0]:
order_items = (
    order_items
    .withColumn("quantity", F.col("quantity").try_cast(IntegerType()))
    .withColumn("unit_price", F.col("unit_price").try_cast(DecimalType(18, 2)))
    .withColumn("discount_pct", F.col("discount_pct").try_cast(DecimalType(5, 2)))
    .withColumn("line_total", F.col("line_total").try_cast(DecimalType(18, 2)))
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("quantity").isNull() | (F.col("quantity") <= 0), F.lit("invalid_quantity")),
            F.when(F.col("unit_price").isNull() | (F.col("unit_price") <= 0), F.lit("invalid_unit_price")),
            F.when(F.col("discount_pct").isNull() | (F.col("discount_pct") < 0) | (F.col("discount_pct") > 100), F.lit("invalid_discount_pct")),
            F.when(F.col("line_total").isNull(), F.lit("invalid_line_total"))
        )
    )
)

### Q29

In [0]:
expected = F.col("quantity") * F.col("unit_price") * (1 - (F.col("discount_pct") / 100))
line_total_ok = F.abs(F.col("line_total") - expected) <= F.lit(0.05)

order_items = (
    order_items
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("dq_reason") == "", F.lit(None)).otherwise(F.col("dq_reason")),
            F.when(~line_total_ok, F.lit("line_total_incorrect"))
        )
    )
)

### Q30

In [0]:
order_keys = (
    spark.read.table("workspace.ecom_silver.orders")
    .select("order_id")
    .distinct()
)

product_keys = (
    spark.read.table("workspace.ecom_silver.products")
    .select("product_id")
    .distinct()
)

order_items = (
    order_items
    .join(
        F.broadcast(order_keys.withColumn("_order_ok", F.lit(True))),
        on = "order_id",
        how = "left",
    )
    .join(
        F.broadcast(product_keys.withColumn("_prod_ok", F.lit(True))),
        on = "product_id",
        how = "left",
    )
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("dq_reason") == "", F.lit(None)).otherwise(F.col("dq_reason")),
            F.when(F.col("_order_ok").isNull(), F.lit("missing_order")),
            F.when(F.col("_prod_ok").isNull(), F.lit("missing_product"))
        )
    )
    .drop("_order_ok", "_prod_ok")
)

In [0]:
# dedpud based on order_item_id
deduped_order_items = (
    order_items.dropDuplicates(["order_item_id"])
)

# save clean and quarantine tables
(
    deduped_order_items
    .filter(F.col("dq_reason").isNull())
    .write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.ecom_silver.order_items")
)

(
    deduped_order_items
    .filter(F.col("dq_reason").isNotNull())
    .write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.ecom_silver.order_items_quarantine")
)

### Q31

In [0]:
method = F.lower(F.trim(F.col("payment_method")))
status = F.lower(F.trim(F.col("payment_status")))
payments = (
    payments
    .withColumn(
        "payment_method",
        F.when(method.isin("credit_card", "debit_card", "credit card", "debit card"), "Card")
         .when(method.isin("cash_on_delivery", "cash on delivery"), "COD")
         .when(method.isin("net_banking", "net banking"), "Net Banking")
         .when(method.isin("wallet"), "Wallet")
         .when(method.isin("upi"), "UPI")
         .otherwise(F.lit(None))
    )
    .withColumn(
        "payment_status",
        F.when(status.isin("success", "successful"), "Success")
         .when(status.isin("failed", "pending", "refunded"), F.initcap(status))
         .otherwise(F.lit(None))  # UNKNOWN
    )
)

In [0]:
orders_totals = (
    spark.table("workspace.ecom_silver.orders")
    .select("order_id", "total_amount")
)
payments = (
    payments
    .withColumn("amount", F.col("amount").try_cast(DecimalType(18, 2)))
    .join(
        F.broadcast(orders_totals.withColumnRenamed("total_amount", "order_total")),
        on="order_id",
        how="left",
    )
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("payment_method").isNull(), F.lit("invalid_payment_method")),
            F.when(F.col("payment_status").isNull(), F.lit("invalid_payment_status")),
            F.when(F.col("amount").isNull(), F.lit("invalid_amount")),
            # successful payment must match order total
            F.when(
                (F.col("payment_status") == "Success")
                & (
                    F.col("order_total").isNull()
                    | (F.abs(F.col("amount") - F.col("order_total")) > F.lit(0.01))
                ),
                F.lit("successful_amount_mismatch"),
            ),
        )
    )
    .drop("order_total")
)

### Q32

In [0]:
order_dates = (
    spark.table("workspace.ecom_silver.orders")
    .select("order_id", "order_date")
)
payments = (
    payments
    .withColumn(
        "payment_date",
        F.expr("try_to_timestamp(payment_date, 'yyyy-MM-dd HH:mm:ss')")
    )
    .join(
        F.broadcast(order_dates.withColumnRenamed("order_date", "order_ts")),
        on="order_id",
        how="left",
    )
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("dq_reason") == "", F.lit(None)).otherwise(F.col("dq_reason")),
            F.when(F.col("payment_date").isNull(), F.lit("invalid_payment_date")),
            F.when(
                F.col("payment_date").isNotNull()
                & F.col("order_ts").isNotNull()
                & (F.col("payment_date") < F.col("order_ts")),
                F.lit("payment_before_order"),
            ),
        )
    )
    .drop("order_ts")
)

In [0]:
# save clean and quarantine tables
deduped_payments = payments.dropDuplicates(["payment_id"])

(
    deduped_payments
    .filter(F.col("dq_reason").isNull())
    .write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.ecom_silver.payments")
)

(
    deduped_payments
    .filter(F.col("dq_reason").isNotNull())
    .write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.ecom_silver.payments_quarantine")
)

### Q33

In [0]:
shipments = (
    shipments
    .withColumn("shipping_cost", F.col("shipping_cost").try_cast(DecimalType(18, 2)))
    .withColumn("shipment_date", F.expr("try_to_date(shipment_date, 'yyyy-MM-dd')"))
    .withColumn("delivery_date", F.expr("try_to_date(delivery_date, 'yyyy-MM-dd')"))
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(
                F.col("shipping_cost").isNull() | (F.col("shipping_cost") < 0),
                F.lit("invalid_shipping_cost"),
            ),
            F.when(
                F.col("tracking_number").isNull()
                | (F.trim(F.col("tracking_number")) == ""),
                F.lit("invalid_tracking_number"),
            ),
            F.when(
                ~F.col("shipment_status").isin("Processing", "In Transit", "Delivered"),
                F.lit("invalid_shipment_status"),
            ),
            F.when(F.col("shipment_date").isNull(), F.lit("invalid_shipment_date")),
            F.when(
                F.col("shipment_date").isNotNull()
                & F.col("delivery_date").isNotNull()
                & (F.col("delivery_date") < F.col("shipment_date")),
                F.lit("invalid_shipment_sequence"),
            ),
        )
    )
)

In [0]:
# save clean and quarantine tables
deduped_shipments = shipments.dropDuplicates(["shipment_id"])

(
    deduped_shipments
    .filter(F.col("dq_reason").isNull())
    .write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.ecom_silver.shipments")
)

(
    deduped_shipments
    .filter(F.col("dq_reason").isNotNull())
    .write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.ecom_silver.shipments_quarantine")
)

### Q34

In [0]:
order_keys = (
    spark.table("workspace.ecom_silver.orders")
    .select("order_id")
    .distinct()
)
product_keys = (
    spark.table("workspace.ecom_silver.products")
    .select("product_id")
    .distinct()
)
returns = (
    returns
    .withColumn("return_date", F.expr("try_to_date(return_date, 'yyyy-MM-dd')"))
    .withColumn("refund_amount", F.col("refund_amount").try_cast(DecimalType(18, 2)))
    .join(
        F.broadcast(order_keys.withColumn("_order_ok", F.lit(True))),
        on="order_id",
        how="left",
    )
    .join(
        F.broadcast(product_keys.withColumn("_prod_ok", F.lit(True))),
        on="product_id",
        how="left",
    )
    .withColumn(
        "dq_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("return_date").isNull(), F.lit("invalid_return_date")),
            F.when(
                F.col("return_reason").isNull()
                | (F.trim(F.col("return_reason")) == ""),
                F.lit("invalid_return_reason"),
            ),
            F.when(
                ~F.col("return_status").isin(
                    "Requested", "Approved", "Refunded", "Rejected"
                ),
                F.lit("invalid_return_status"),
            ),
            # refund: must cast cleanly and be nonnegative (data has -10)
            F.when(
                F.col("refund_amount").isNull() | (F.col("refund_amount") < 0),
                F.lit("invalid_refund_amount"),
            ),
            # references must exist in valid Silver
            F.when(F.col("_order_ok").isNull(), F.lit("missing_order")),
            F.when(F.col("_prod_ok").isNull(), F.lit("missing_product")),
        )
    )
    .drop("_order_ok", "_prod_ok")
)

In [0]:
deduped_returns = returns.dropDuplicates(["return_id"])

(
    deduped_returns
    .filter(F.col("dq_reason").isNull())
    .write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.ecom_silver.returns")
)

(
    deduped_returns
    .filter(F.col("dq_reason").isNotNull())
    .write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.ecom_silver.returns_quarantine")
)

### Q35

In [0]:
records = []
tables = ["customers", "orders", "order_items", "payments", "products", "shipments", "returns"]
for table in tables:
    bronze = spark.read.format("delta").table(f"workspace.ecom_bronze.{table}_raw")
    silver = spark.read.format("delta").table(f"workspace.ecom_silver.{table}")
    records.append(
        {
            "table_name": table,
            "silver_rows": silver.count(),
            "bronze_rows": bronze.count(),
            'valid_pct': round(silver.count() / bronze.count() * 100, 2)
        }
    )

results = spark.createDataFrame(records).select("table_name", "silver_rows", "bronze_rows", "valid_pct")
display(results)

table_name,silver_rows,bronze_rows,valid_pct
customers,5632,6000,93.87
orders,0,25000,0.0
order_items,0,62053,0.0
payments,0,25000,0.0
products,1084,1200,90.33
shipments,0,20000,0.0
returns,0,3000,0.0


## Part E - Gold and Delta Lake

### Q56

The assignment uses built-in functions instead of Python UDFs to capitalize on the Spark's optimization planning in the lazy evaluation of transformations before actions are called.

### Q57

Invalid records are saved to a quarantine table instead of being silently deleted so they can be investigate. After investigation, the records may prove to be malformed and not useful for analytics (in which case they then might get silently deleted), or they can be referenced from other sources and potentially updated to then be included in the clean tables.

### Q58 

Bronze values are initially stored as strings because that's the default data type when data is loaded without any schema to inform specific data types.

### Q59 

The notebook doesn't use RDD APIs because the data is structured with some sense of what data type each column should be. It's more optimal to use the built-in Spark functions rather than performing the same steps using a low-level data abstraction such as a RDD.

### Q60